# Charcoal hints (柳炭) for Splashed Ink

A prototype procedure. You splash ink in Splashed Ink, export the paper, and this notebook suggests what each blot could be, as faint lines like a willow-charcoal underdrawing. You load the suggestions back into Splashed Ink, where they fade away. You decide: follow a suggestion, ignore it, or see something else, and draw it yourself.

Two kinds of suggestion, side by side for comparison:

* **Generated.** Stable Diffusion 1.5 with a small LoRA trained only on 587 motifs from the *Mustard Seed Garden Manual* (1935 edition, National Diet Library, public domain) starts from your blot and draws what it sees in it (img2img). The base model was trained on web images; the LoRA only steers it toward the manual's motifs and line.
* **Retrieved.** The manual motif whose silhouette best fits the blot, stretched into it. Only the manual is involved.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Training the LoRA takes about 40–60 minutes and only has to be done once if you save it to Google Drive (step 2).

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install diffusers==0.31.0 transformers==4.46.3 accelerate==1.1.1 peft==0.13.2 datasets==3.1.0 huggingface_hub==0.26.2 ftfy safetensors
!git clone -q --depth 1 https://github.com/Lee-arts-02/Splashed-Ink.git repo
!git clone -q --depth 1 -b lora-data https://github.com/Lee-arts-02/Splashed-Ink.git lora-data
!wget -q -O train_lora.py https://raw.githubusercontent.com/huggingface/diffusers/v0.31.0/examples/text_to_image/train_text_to_image_lora.py
import sys; sys.path.insert(0, 'repo/tools')
import charcoal as C
print(len(open('lora-data/data/metadata.jsonl').readlines()), 'training images')

## 2. Where to keep the LoRA

Mount Google Drive so the trained LoRA survives the session. If you skip this, it is kept only until the runtime ends.

In [ ]:
USE_DRIVE = True   # set to False to keep everything in this session only
import os
if USE_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    LORA_DIR = '/content/drive/MyDrive/splashed-ink/lora-jieziyuan'
else:
    LORA_DIR = '/content/lora-jieziyuan'
os.makedirs(LORA_DIR, exist_ok=True)
TRAINED = os.path.exists(os.path.join(LORA_DIR, 'pytorch_lora_weights.safetensors'))
print('LoRA already trained:' , TRAINED, LORA_DIR)

## 3. Train the LoRA (once)

Skipped automatically if a trained LoRA is already in `LORA_DIR`. About 1,500 steps; watch the loss fall. Validation images of a bird appear in the output folder every few epochs.

In [ ]:
import subprocess, sys
if not TRAINED:
    cmd = ['accelerate', 'launch', '--mixed_precision=fp16', 'train_lora.py',
           '--pretrained_model_name_or_path=stable-diffusion-v1-5/stable-diffusion-v1-5',
           '--train_data_dir=lora-data/data', '--caption_column=text',
           '--resolution=512', '--random_flip', '--train_batch_size=4', '--gradient_checkpointing',
           '--max_train_steps=1500', '--learning_rate=1e-4', '--lr_scheduler=cosine', '--lr_warmup_steps=50',
           '--rank=16', '--seed=42', '--checkpointing_steps=500', f'--output_dir={LORA_DIR}',
           '--validation_prompt=jieziyuan, woodblock line drawing of a bird or flower, black ink lines on white paper',
           '--num_validation_images=2', '--validation_epochs=20']
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    import re; last = -1
    for line in p.stdout:                       # progress every 100 steps, and any error in full
        m = re.search(r'Steps:.*?(\d+)/(\d+)', line)
        if m:
            s = int(m.group(1))
            if s // 100 != last: last = s // 100; print(line.strip()[:160])
        elif 'Error' in line or 'Traceback' in line or 'error:' in line: print(line.rstrip())
    p.wait(); print('exit code', p.returncode)
TRAINED = os.path.exists(os.path.join(LORA_DIR, 'pytorch_lora_weights.safetensors')); print('trained:', TRAINED)

## 4. Your paper

In Splashed Ink, splash some ink and press **Export paper** (in the Session panel). Upload that PNG here. If you upload nothing, the example sheet from the repository is used.

In [ ]:
from google.colab import files
up = files.upload()
PAPER = next(iter(up)) if up else 'repo/examples/paper-example.png'
T, paper = C.load_paper(PAPER); blots = C.find_blots(T)[:5]
print(paper.size, len(blots), 'blots'); paper

## 5. Readings

`STRENGTH` is how far the model may wander from your ink: low keeps close to the blot's shape (and sees little in it); high draws a whole motif that may cover over what you would have seen yourself. That distance is itself a setting of friction worth trying at several values.

Each run makes `N_READINGS` different readings of every blot (different seeds and subjects), plus retrieved readings for comparison.

In [ ]:
import torch, numpy as np, json
from diffusers import StableDiffusionImg2ImgPipeline
pipe = StableDiffusionImg2ImgPipeline.from_pretrained('stable-diffusion-v1-5/stable-diffusion-v1-5', torch_dtype=torch.float16, safety_checker=None).to('cuda')
pipe.load_lora_weights(LORA_DIR)
SUBJECTS = ['bird', 'fish', 'rock', 'orchid', 'bamboo', 'plum blossom branch', 'chrysanthemum', 'figure', 'animal', 'flowering plant', 'insect', 'building']
idx = C.MotifIndex('lora-data/data')

In [ ]:
STRENGTH = 0.62; N_READINGS = 3; GUIDANCE = 6.5; STEPS = 30; SEED = 0
rng = np.random.default_rng(SEED); gen_layers, ret_layers, notes = [], [], []
for r in range(N_READINGS):
    gm = np.zeros(T.shape, np.float32); rm = np.zeros(T.shape, np.float32); note = []
    for b in blots:
        crop, cbox = C.square_crop(T, b['box'])
        subject = SUBJECTS[rng.integers(len(SUBJECTS))]; seed = int(rng.integers(1e9))
        img = pipe(prompt=f'jieziyuan, woodblock line drawing of a {subject}, black ink lines on white paper',
                   negative_prompt='color, shading, photograph, text, frame', image=C.to_model_image(crop),
                   strength=STRENGTH, guidance_scale=GUIDANCE, num_inference_steps=STEPS,
                   generator=torch.Generator('cuda').manual_seed(seed)).images[0]
        gm = np.maximum(gm, C.place(C.extract_lines(img, crop), cbox, T.shape))
        pick = idx.best(b['mask'], k=4, rng=rng); rm = np.maximum(rm, idx.render(pick, b['box'], T.shape))
        note.append({'blot': b['box'], 'generated': {'subject': subject, 'seed': seed, 'strength': STRENGTH},
                     'retrieved': {'motif': idx.items[pick[1]]['meta']['file_name'], 'source': idx.items[pick[1]]['meta']['source'], 'fit': round(float(pick[0]), 3)}})
    gen_layers.append(C.charcoal_png(gm)); ret_layers.append(C.charcoal_png(rm)); notes.append(note)
display(C.preview(paper, gen_layers)); display(C.preview(paper, ret_layers))

## 6. Take the charcoal back to the paper

Download the layers. In Splashed Ink, **Load charcoal** (Session panel) and pick one: it appears over the paper and fades. `readings.json` says what each blot was read as; keep it for the session record.

In [ ]:
import zipfile
os.makedirs('out', exist_ok=True)
for i, L in enumerate(gen_layers): L.save(f'out/charcoal-generated-{i+1}.png')
for i, L in enumerate(ret_layers): L.save(f'out/charcoal-retrieved-{i+1}.png')
json.dump({'paper': PAPER, 'strength': STRENGTH, 'readings': notes}, open('out/readings.json', 'w'), indent=1)
with zipfile.ZipFile('charcoal.zip', 'w') as z:
    for f in os.listdir('out'): z.write('out/' + f, f)
files.download('charcoal.zip')